# Fix Label Tambang Papua — re-export label (union +Maus) untuk 2 tile terdampak + patch in-place

**Konteks.** `qc_label_tambang_papua.ipynb` menemukan: label Tambang test/train sekarang murni
Tang & Werner 2023 (patch sudah dipotong sebelum union +Maus 2022 ditambahkan ke kode), gap
+38,7% area. Tile Papua yang overlap poligon Maus HANYA **2 dari 36**: `papua_t2_tile_21`
(area Grasberg/Tembagapura) dan `papua_t2_tile_32` (tenggara).

**Kenapa tidak generate ulang `split_files()`?** Val/test TIDAK punya manifest per-file —
assignment-nya cuma fisik (file sudah masuk `val.tar`/`test.tar` di `Bahan_Training_Fix`).
`split_files()` pakai `rng.permutation(seed=42)` atas urutan list file SAAT itu — kalau
dijalankan ulang dengan jumlah/urutan file yang sedikit berbeda, peta index->file bisa BERGESER
TOTAL untuk file-file LAIN juga (bukan cuma yang baru). Itu bisa diam-diam mengubah identitas
test-set yang sudah jadi acuan `metrics.json` — risikonya kebocoran/pergeseran split.

**Solusi aman (dipakai di sini): patch `lab` IN-PLACE.** Tiap `.npz` di dalam tar menyimpan
`tile`/`row`/`col` (lihat `cut_patches()` di `patches.py`). Untuk patch yang `tile`-nya salah
satu dari 2 tile terdampak: hitung ulang `lab` dari label GEE yang sudah di-refresh, di window
piksel `row:row+256, col:col+256` yang SAMA. `img` dan lokasi split (train/val/test) TIDAK
berubah sama sekali.

**Desain aman:**
- Hasil surgery ditulis ke folder **BARU** (`Bahan_Training_Fix_LabelFix/`), TIDAK menimpa
  `Bahan_Training_Fix` asli — supaya data yang sudah dipakai utk `metrics.json` model 1/2/3
  sekarang tidak tersentuh sampai kamu yakin hasilnya benar.
- `DRY_RUN=True` default di cell surgery — cuma melaporkan berapa patch akan berubah,
  TIDAK menulis apa pun, sampai kamu sengaja set `False`.
- Re-export GEE HANYA band `label` (bukan citra Sentinel-2 yang tidak berubah) untuk 2 tile
  saja — ringan & cepat dibanding re-export penuh.

In [ ]:
# === SETUP (Colab) — clone repo + install + mount Drive ===
import sys, subprocess, importlib
from pathlib import Path

subprocess.run(
    "cd /content && (git -C fw_repo pull -q || git clone --depth 1 "
    "https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo)",
    shell=True, check=False,
)
subprocess.run("pip install -q -e /content/fw_repo[gee,gis,ml]", shell=True, check=False)
if "/content/fw_repo/src" not in sys.path:
    sys.path.insert(0, "/content/fw_repo/src")
for _m in [m for m in list(sys.modules) if m == "forestwatch" or m.startswith("forestwatch.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/Satria Data 3.0")
print("Setup selesai. DRIVE_ROOT:", DRIVE_ROOT)

In [ ]:
# === AUTH GEE + KONFIGURASI + 2 TILE TERDAMPAK (dari qc_label_tambang_papua.ipynb) ===
import ee
from forestwatch.gee.auth import init_ee
from forestwatch.constants import PAPUA_BBOX
from forestwatch.config import load_config
from forestwatch.gee.tiles import make_tiles_from_bbox

cfg = load_config()
init_ee(project=cfg["project"]["gee_project_id"])
T2 = cfg["periods"]["t2"]

NX, NY = cfg["export"]["tiles_nx"], cfg["export"]["tiles_ny"]
tile_bboxes = make_tiles_from_bbox(PAPUA_BBOX, nx=NX, ny=NY)

# Hasil qc_label_tambang_papua.ipynb: index 21 & 32 overlap poligon Maus.
AFFECTED_IDX = [21, 32]
for idx in AFFECTED_IDX:
    print(f"papua_t2_tile_{idx:02d}.tif  bbox={tile_bboxes[idx]}")

In [ ]:
# === RE-EXPORT LABEL SAJA (union +Maus, kode build_label() sekarang) utk 2 tile ===
from forestwatch.gee.label_fusion import build_label
from forestwatch.gee.export import export_stack

lf = cfg["label_fusion"]
LABEL_FIX_FOLDER = "Label_Fix_Tambang_Maus"  # subfolder baru di Drive root

tasks = []
for idx in AFFECTED_IDX:
    xmin, ymin, xmax, ymax = tile_bboxes[idx]
    tile_geom = ee.Geometry.Rectangle([xmin, ymin, xmax, ymax])
    label_new = build_label(
        tile_geom, T2,
        hansen_loss_year_min=lf["hansen_loss_year_min"],
        hansen_erosion_pixels=lf["hansen_erosion_pixels"],
        palm_prob_threshold=lf["palm_prob_threshold"],
        esa_forest_union=lf.get("esa_forest_union", False),
    )  # build_label sudah .toByte().clip(region) -- sama persis spt Bagian 7 full_pipeline
    desc = f"papua_t2_tile_{idx:02d}_label_fix"
    task = export_stack(
        label_new, description=desc, folder=LABEL_FIX_FOLDER,
        region=tile_geom, scale=cfg["sentinel2"]["scale"],
    )
    tasks.append((desc, task))

print("Submitted", len(tasks), "task export ke folder Drive:", LABEL_FIX_FOLDER)
for desc, _ in tasks:
    print(" -", desc)
print()
print("Pantau di https://code.earthengine.google.com/tasks -- TUNGGU status COMPLETED")
print("keduanya sebelum lanjut ke cell berikutnya (band tunggal, 2 tile -- harusnya cepat).")

In [ ]:
# === VERIFIKASI: GeoTIFF label baru sudah ada di Drive (jalankan SETELAH task COMPLETED) ===
# PENTING: ee.batch.Export.image.toDrive(folder=...) SELALU bikin folder di ROOT 'My Drive',
# bukan nested di DRIVE_ROOT ('Satria Data 3.0') -- sama spt semua ekspor lain di proyek ini.
# Drive FUSE mount Colab kadang lag sinkronisasi (sudah pernah terjadi sebelumnya di proyek
# ini) -- isi folder baru/baru-dipindah belum ter-refresh di cache mount walau sudah benar di
# web UI. Retry dgn paksa listdir() dulu (trigger refresh) sebelum nyerah.
import time

MY_DRIVE_ROOT = Path("/content/drive/MyDrive")
LABEL_FIX_DIR = MY_DRIVE_ROOT / LABEL_FIX_FOLDER

AFFECTED_TILE_FILES = {}
for attempt in range(6):
    try:
        _ = list(LABEL_FIX_DIR.iterdir())  # paksa refresh listing direktori (trigger sync FUSE)
    except FileNotFoundError:
        _ = []
    AFFECTED_TILE_FILES = {}
    for idx in AFFECTED_IDX:
        tile_name = f"papua_t2_tile_{idx:02d}.tif"
        fix_path = LABEL_FIX_DIR / f"papua_t2_tile_{idx:02d}_label_fix.tif"
        if fix_path.exists():
            AFFECTED_TILE_FILES[tile_name] = fix_path
    if len(AFFECTED_TILE_FILES) == len(AFFECTED_IDX):
        break
    print(f"  (percobaan {attempt + 1}/6) belum lengkap, isi direktori saat ini:", [p.name for p in _])
    time.sleep(5)

for idx in AFFECTED_IDX:
    tile_name = f"papua_t2_tile_{idx:02d}.tif"
    status = "OK" if tile_name in AFFECTED_TILE_FILES else "BELUM ADA setelah retry"
    print(f"{tile_name:<28} -> papua_t2_tile_{idx:02d}_label_fix.tif  {status}")

assert len(AFFECTED_TILE_FILES) == len(AFFECTED_IDX), (
    "Masih belum lengkap setelah retry -- coba Runtime > Restart session lalu mount ulang Drive "
    "(force_remount=True), atau pastikan nama folder/file persis sama (cek typo di Drive)."
)


## Diagnostik: 0 patch terdampak itu mencurigakan -- cek format nama 'tile' sebenarnya

Mustahil 0 dari 77.051 patch berasal dari `papua_t2_tile_21`/`_32` kalau memang kedua tile
itu punya piksel Tambang (sudah dikonfirmasi di `qc_label_tambang_papua.ipynb`). Kemungkinan
besar nama tile asli yang tersimpan di field `tile` setiap `.npz` BERBEDA format dari yang
saya asumsikan (`papua_t2_tile_21.tif`) -- cek dulu sample asli sebelum lanjut.

In [ ]:
# === DIAGNOSTIK: print sample nilai field 'tile' asli dari beberapa .npz ===
import io, tarfile, collections
import numpy as np

sample_tar = BAHAN_DIR / 'train' / 'train_part01.tar'
tile_counter = collections.Counter()
n_checked = 0
n_has_tile_field = 0
with tarfile.open(sample_tar, 'r') as t:
    for member in t.getmembers()[:500]:  # sample 500 patch pertama, cukup utk lihat pola
        raw = t.extractfile(member).read()
        data = np.load(io.BytesIO(raw))
        n_checked += 1
        if 'tile' in data.files:
            n_has_tile_field += 1
            tile_counter[str(data['tile'])] += 1

print(f'Dicek {n_checked} patch dari {sample_tar.name}, {n_has_tile_field} punya field "tile".')
print()
print('Sample nilai unik field "tile" yang ditemukan (sampai 15):')
for name, count in tile_counter.most_common(15):
    print(f'  {name!r}  (x{count})')
print()
print('Bandingkan format ini dengan yang diasumsikan kode surgery:', list(AFFECTED_TILE_FILES.keys()))


In [ ]:
# === SURGERY: patch 'lab' IN-PLACE di tar Bahan_Training_Fix, output ke folder BARU ===
# DRY_RUN=True (default): cuma laporan, TIDAK menulis apa pun ke Drive.
# Set False HANYA setelah preview di bawah terlihat wajar (jumlah patch & split masuk akal).
DRY_RUN = True

import io, tarfile
import numpy as np
import rasterio
from tqdm.auto import tqdm

PATCH_SIZE = cfg["patches"]["size"]
BAHAN_DIR = DRIVE_ROOT / "Bahan_Training_Fix"
OUT_DIR = DRIVE_ROOT / "Bahan_Training_Fix_LabelFix"  # folder BARU, asli tak tersentuh

new_label_rasters = {}
for tile_name, path in AFFECTED_TILE_FILES.items():
    with rasterio.open(path) as src:
        new_label_rasters[tile_name] = src.read(1).astype("uint8")
    print("Label baru dimuat:", tile_name, "shape =", new_label_rasters[tile_name].shape)


import re

# GEE memecah tile besar jadi beberapa file SHARD saat ekspor asli (krn ukuran kelewat
# besar utk 1 file) -- nama jadi 'papua_t2_tile_21-0000000000-0000012544.tif', dgn
# suffix '-{x_offset:010d}-{y_offset:010d}' = offset piksel shard itu dlm tile PENUH.
# row/col di .npz relatif ke shard sendiri -- harus ditambah offset ini dulu sebelum
# diindex ke raster baru (yg di-export ulang sbg 1 file utuh, tanpa shard).
_TILE_RE = re.compile(r'^(papua_t2_tile_\d+)(?:-(\d+)-(\d+))?\.tif$')


def patch_npz_bytes(raw_bytes):
    """Return (bytes_baru, changed). lab diganti kalau 'tile' termasuk yang terdampak.

    Sebagian patch kehilangan field 'tile'/'row'/'col' krn pernah ditimpa Bagian 10B
    Data Healing (np.savez_compressed(p, img=img, lab=lab) -- cuma simpan img+lab, buang
    metadata). Patch begini TIDAK BISA diidentifikasi asal tile-nya -- aman dilewati
    (bukan target tile_21/_32 yg kita tahu pasti, jadi skip != salah).
    """
    data = np.load(io.BytesIO(raw_bytes))
    if "tile" not in data.files:
        return raw_bytes, False
    tile_raw = str(data["tile"])
    m = _TILE_RE.match(tile_raw)
    if not m:
        return raw_bytes, False
    base_tile = m.group(1) + ".tif"  # nama dasar tanpa suffix shard
    if base_tile not in new_label_rasters:
        return raw_bytes, False
    x_off = int(m.group(2)) if m.group(2) else 0
    y_off = int(m.group(3)) if m.group(3) else 0
    row, col = int(data["row"]) + y_off, int(data["col"]) + x_off
    full = new_label_rasters[base_tile]
    new_lab = full[row:row + PATCH_SIZE, col:col + PATCH_SIZE]
    if new_lab.shape != (PATCH_SIZE, PATCH_SIZE):
        return raw_bytes, False  # di luar batas raster baru -- aman, skip
    out = io.BytesIO()
    np.savez(out, img=data["img"], lab=new_lab, tile=data["tile"], row=data["row"], col=data["col"])
    return out.getvalue(), True


def process_tar(tar_path, out_path):
    changed, total = 0, 0
    if not DRY_RUN:
        out_path.parent.mkdir(parents=True, exist_ok=True)
    mode_out = "w" if not DRY_RUN else None
    tout = tarfile.open(out_path.with_suffix(".tar.partial"), mode_out) if not DRY_RUN else None
    with tarfile.open(tar_path, "r") as tin:
        for member in tin.getmembers():
            total += 1
            raw = tin.extractfile(member).read()
            new_raw, did_change = patch_npz_bytes(raw)
            if did_change:
                changed += 1
            if not DRY_RUN:
                info = tarfile.TarInfo(name=member.name)
                info.size = len(new_raw)
                info.mtime = member.mtime
                tout.addfile(info, io.BytesIO(new_raw))
    if not DRY_RUN:
        tout.close()
        out_path.with_suffix(".tar.partial").rename(out_path)  # atomic
    return changed, total


tar_files = []
for split in ["train", "val", "test"]:
    split_dir = BAHAN_DIR / split
    if split_dir.exists():
        tar_files += [(split, p) for p in sorted(split_dir.glob("*.tar"))]

print(f"{'[DRY RUN] ' if DRY_RUN else ''}Memproses {len(tar_files)} tar dari {BAHAN_DIR}...")
summary = {}
for split, tp in tqdm(tar_files, desc="Tar"):
    out_p = OUT_DIR / split / tp.name
    changed, total = process_tar(tp, out_p)
    summary.setdefault(split, [0, 0])
    summary[split][0] += changed
    summary[split][1] += total
    print(f"  {split}/{tp.name}: {changed}/{total} patch terdampak")

print()
print("RINGKASAN per split:")
for split, (changed, total) in summary.items():
    print(f"  {split}: {changed} patch diganti labelnya (dari {total} total)")
if DRY_RUN:
    print()
    print("Ini DRY RUN -- tidak ada file ditulis. Kalau ringkasan di atas wajar, set")
    print("DRY_RUN=False lalu jalankan ulang cell ini utk eksekusi nyata ke", OUT_DIR)

## Setelah `DRY_RUN=False` berhasil

1. `Bahan_Training_Fix_LabelFix/{train,val,test}/*.tar` berisi salinan PENUH dataset dengan label
   Tambang ter-refresh untuk 2 tile terdampak -- `Bahan_Training_Fix` asli TIDAK berubah.
2. Copy juga `train_rajaampat/`, `patch_sampler_weights_shared.json`, `class_weights.json` dari
   `Bahan_Training_Fix` asli ke `Bahan_Training_Fix_LabelFix` (tidak terdampak, tapi training
   notebook butuh semuanya ada di folder yang sama).
3. Hitung ulang `class_weights.json` & `patch_sampler_weights_shared.json` HANYA bila perlu --
   perubahan 2 tile dari 36 kemungkinan tidak signifikan mengubah distribusi kelas global, tapi
   cek dulu di `optimize_dataset.ipynb` sebelum asumsi aman dilewati.
4. Setelah yakin, baru putuskan: ganti nama folder (`Bahan_Training_Fix` -> `_old`, lalu
   `Bahan_Training_Fix_LabelFix` -> `Bahan_Training_Fix`) supaya notebook training existing
   otomatis pakai data baru tanpa ubah path di notebook training.
5. Re-evaluasi `metrics.json` test set model 1 dgn data yang sudah di-refresh (tanpa training
   ulang -- checkpoint sama, cuma ganti `test_loader` baca dari folder baru) untuk lihat
   seberapa besar Tambang IoU naik MURNI dari perbaikan label, sebelum fine-tune loss-function.

In [ ]:
# === LANGKAH 2: copy train_rajaampat/ + json pendukung ke folder baru ===
# Surgery cuma menulis train/val/test/*.tar -- 3 item ini TIDAK tersentuh tapi notebook
# training butuh semuanya ada di folder yang sama (Bahan_Training_Fix_LabelFix).
import shutil

EXTRA_ITEMS = ["train_rajaampat", "class_weights.json", "patch_sampler_weights_shared.json"]

for item in EXTRA_ITEMS:
    src = BAHAN_DIR / item
    dst = OUT_DIR / item
    if not src.exists():
        print(f"  [skip] {item}: tidak ada di {BAHAN_DIR}")
        continue
    if dst.exists():
        print(f"  [skip] {item}: sudah ada di {OUT_DIR}")
        continue
    if src.is_dir():
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    print(f"  [OK] {item} disalin ke {dst}")

print()
print("Isi", OUT_DIR, "sekarang:")
for p in sorted(OUT_DIR.iterdir()):
    print("  -", p.name)

## Langkah 3: cek dampak ke distribusi kelas global (sebelum asumsi aman dilewati)

Hitung histogram piksel per-kelas HANYA untuk 1.546 patch yang berubah (label LAMA dari
`Bahan_Training_Fix` asli vs label BARU yang sudah dipatch) -- jauh lebih cepat dari scan
ulang 77.051 patch penuh, karena sisanya identik (tak perlu dihitung ulang).

In [ ]:
# === LANGKAH 3: estimasi pergeseran distribusi kelas global (LAMA vs BARU) ===
import io, tarfile
import numpy as np
from forestwatch.constants import N_CLASSES, CLASS_NAMES

old_hist = np.zeros(N_CLASSES, dtype=np.int64)
new_hist = np.zeros(N_CLASSES, dtype=np.int64)
n_affected_seen = 0

for split in ["train", "val", "test"]:
    split_dir = BAHAN_DIR / split
    if not split_dir.exists():
        continue
    for tar_path in sorted(split_dir.glob("*.tar")):
        with tarfile.open(tar_path, "r") as tin:
            for member in tin.getmembers():
                raw = tin.extractfile(member).read()
                data = np.load(io.BytesIO(raw))
                if "tile" not in data.files:
                    continue
                m = _TILE_RE.match(str(data["tile"]))
                if not m:
                    continue
                base_tile = m.group(1) + ".tif"
                if base_tile not in new_label_rasters:
                    continue
                x_off = int(m.group(2)) if m.group(2) else 0
                y_off = int(m.group(3)) if m.group(3) else 0
                row, col = int(data["row"]) + y_off, int(data["col"]) + x_off
                new_lab = new_label_rasters[base_tile][row:row + PATCH_SIZE, col:col + PATCH_SIZE]
                if new_lab.shape != (PATCH_SIZE, PATCH_SIZE):
                    continue
                old_hist += np.bincount(data["lab"].ravel(), minlength=N_CLASSES)
                new_hist += np.bincount(new_lab.ravel(), minlength=N_CLASSES)
                n_affected_seen += 1

print(f"Patch terdampak yang berhasil dihitung: {n_affected_seen}")
print()
TOTAL_PATCHES_DATASET = 38360 + 19345 + 19346  # train + val + test (dari ringkasan surgery)
TOTAL_PIXELS_DATASET = TOTAL_PATCHES_DATASET * PATCH_SIZE * PATCH_SIZE
print(f"{'Kelas':<16} {'Lama (px)':>14} {'Baru (px)':>14} {'Delta (px)':>12} {'Delta (% total dataset)':>24}")
for c in range(N_CLASSES):
    delta = int(new_hist[c]) - int(old_hist[c])
    pct = delta / TOTAL_PIXELS_DATASET * 100
    print(f"{CLASS_NAMES[c]:<16} {int(old_hist[c]):>14,} {int(new_hist[c]):>14,} {delta:>12,} {pct:>23.4f}%")
print()
print("Kalau semua |delta (% total dataset)| di atas << 0,1% -- aman lewati hitung ulang")
print("class_weights.json. Kalau ada kelas (terutama Tambang) bergeser signifikan, jalankan")
print("ulang sel hitung distribusi kelas di optimize_dataset.ipynb dengan path folder baru.")

## Langkah 4: swap nama folder (HANYA setelah yakin langkah 1-3 di atas oke)

`SWAP_CONFIRM=False` default -- gerbang pengaman yang sama spt `DRY_RUN`, supaya tidak ada
rename tidak sengaja. Rename di Drive (FUSE) cuma operasi metadata, bukan copy ulang 40GB --
cepat. `Bahan_Training_Fix` asli TIDAK dihapus, cuma diganti nama jadi `_old` (bisa
dikembalikan kapan saja kalau ternyata ada yang salah).

In [ ]:
# === LANGKAH 4: swap nama folder -- Bahan_Training_Fix <-> Bahan_Training_Fix_LabelFix ===
SWAP_CONFIRM = False  # set True HANYA setelah yakin (langkah 1-3 sudah oke)

OLD_BACKUP_DIR = DRIVE_ROOT / "Bahan_Training_Fix_old"

if not SWAP_CONFIRM:
    print("SWAP_CONFIRM=False -- tidak melakukan apa pun. Set True untuk eksekusi nyata.")
    print(f"Rencana: {BAHAN_DIR.name} -> {OLD_BACKUP_DIR.name}")
    print(f"         {OUT_DIR.name} -> {BAHAN_DIR.name}")
else:
    assert not OLD_BACKUP_DIR.exists(), f"{OLD_BACKUP_DIR} sudah ada -- hapus/rename manual dulu."
    BAHAN_DIR.rename(OLD_BACKUP_DIR)
    OUT_DIR.rename(BAHAN_DIR)
    print(f"OK: {BAHAN_DIR.name} (lama) -> {OLD_BACKUP_DIR.name}")
    print(f"OK: Bahan_Training_Fix_LabelFix -> {BAHAN_DIR.name} (aktif sekarang)")
    print()
    print("Notebook training existing otomatis pakai data baru (path sama). Kalau ternyata")
    print(f"ada yang salah, kembalikan: rename {OLD_BACKUP_DIR.name} -> {BAHAN_DIR.name} lagi")
    print("(setelah hapus/rename dulu folder LabelFix yg sudah jadi Bahan_Training_Fix).")